# Member 3: Q5 CornersProblem (Problem Representation)
**Student Name:** Gajanayake H.K.D.W.R  
**Student ID:** IT24510235  
**Group:** Group_AI_28  
**Module:** SE3062 - Intelligent Systems  

---
### 📌 Overview
Implementation of the `CornersProblem` class in `searchAgents.py` (8 Marks).
The objective is finding a path that touches all four corners of the maze.

### State Space Design
* **State:** `((x, y), (c1_visited, c2_visited, c3_visited, c4_visited))`
* **Why?** Compact, immutable, and hashable tuple allowing $O(1)$ lookups in `expanded` set without storing entire GameState or Walls grid.


---
## ⚙️ Step 0: Setup & Autograder Runner (RUN THIS FIRST)
This cell automatically finds where `autograder.py` is located (whether running in **VS Code**, **Jupyter**, or **Google Colab**), adds the search framework to `sys.path` to enable all imports, and defines a universal `run_autograder()` function so all tests execute without `FileNotFoundError`.


In [ ]:
# 🚀 Universal Autograder Runner (VS Code / Local / Colab)
import os
import sys
import subprocess

def find_search_dir():
    current = os.path.abspath(os.getcwd())
    
    # 1. Search parent directories
    d = current
    for _ in range(6):
        c1 = os.path.join(d, "Group_Project", "search")
        if os.path.exists(os.path.join(c1, "autograder.py")):
            return os.path.abspath(c1)
        c2 = os.path.join(d, "search")
        if os.path.exists(os.path.join(c2, "autograder.py")):
            return os.path.abspath(c2)
        if os.path.exists(os.path.join(d, "autograder.py")):
            return os.path.abspath(d)
        p = os.path.dirname(d)
        if p == d:
            break
        d = p
        
    # 2. In Google Colab (/content)
    if os.path.exists("/content"):
        for root, dirs, files in os.walk("/content"):
            if "autograder.py" in files and "test_cases" in dirs:
                return root
        # If not cloned yet in Colab, clone from GitHub
        print("📥 Cloning repository in Colab...")
        subprocess.run(["git", "clone", "https://github.com/IT24400002/SE3062-assignment_1.git"], cwd="/content")
        for root, dirs, files in os.walk("/content"):
            if "autograder.py" in files and "test_cases" in dirs:
                return root

    return None

SEARCH_DIR = find_search_dir()

if SEARCH_DIR:
    if SEARCH_DIR not in sys.path:
        sys.path.insert(0, SEARCH_DIR)
    try:
        os.chdir(SEARCH_DIR)
        print(f"✅ Ready! Autograder located at: {SEARCH_DIR}")
    except Exception as e:
        print(f"✅ Ready! (Using subprocess in {SEARCH_DIR})")
else:
    print("❌ Could not locate autograder.py directory.")

def run_autograder(question=None):
    if not SEARCH_DIR or not os.path.exists(os.path.join(SEARCH_DIR, "autograder.py")):
        print("❌ autograder.py not found! Please check your directory structure.")
        return
        
    cmd = [sys.executable, "autograder.py"]
    if question:
        cmd.extend(["-q", question])
    print(f"📍 Running autograder in: {SEARCH_DIR}\n")
    proc = subprocess.Popen(cmd, cwd=SEARCH_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()


---
## 📦 Framework Imports


In [ ]:
# Safe imports enabled by Step 0
import search
import pacman
from game import Directions, Actions
from typing import Any
print("search, pacman, Directions, and Actions imported successfully!")


---
## Q5: CornersProblem Implementation


In [ ]:
class CornersProblem(search.SearchProblem):
    """
    Search problem finding paths through all four corners of a layout.
    """
    def __init__(self, startingGameState: pacman.GameState):
        # 1. Store maze walls and starting position
        self.walls = startingGameState.getWalls()
        self.startingPosition = startingGameState.getPacmanPosition()
        top, right = self.walls.height - 2, self.walls.width - 2
        # 2. Define coordinates for all 4 corners
        self.corners = ((1, 1), (1, top), (right, 1), (right, top))
        for corner in self.corners:
            if not startingGameState.hasFood(*corner):
                print('Warning: no food in corner ' + str(corner))
        self._expanded = 0 # Number of search nodes expanded

    def getStartState(self):
        # State: ((x, y), (c1_visited, c2_visited, c3_visited, c4_visited))
        # Check if the starting position is already on any corner
        visited = tuple(self.startingPosition == c for c in self.corners)
        return (self.startingPosition, visited)

    def isGoalState(self, state: Any):
        # Goal is reached when all 4 corners are True (visited)
        return all(state[1])

    def getSuccessors(self, state: Any):
        successors = []
        # 1. Try moving in all 4 cardinal directions
        for action in [Directions.NORTH, Directions.SOUTH, Directions.EAST, Directions.WEST]:
            (x, y), visited = state
            dx, dy = Actions.directionToVector(action)
            nextx, nexty = int(x + dx), int(y + dy)
            # 2. Check if next position is free (not a wall)
            if not self.walls[nextx][nexty]:
                nextPos = (nextx, nexty)
                # 3. Update visited corners: True if already visited or newly reached
                newVisited = tuple(v or nextPos == c for v, c in zip(visited, self.corners))
                # 4. Successor triple: (new_state, action, stepCost=1)
                successors.append(((nextPos, newVisited), action, 1))

        self._expanded += 1  # Track search node expansions
        return successors


---
## 🧪 Autograder Test Cell


In [ ]:
# Test Q5 - Corners Problem (Score: 3/3 | 8 Marks)
run_autograder('q5')
